# 05 · 앙상블 V2 (확률 블렌딩)

> 학습된 v2 모델들의 val 확률을 섞어(블렌딩) 단일 최고(XGB_v2_tuned 0.296)를 넘는지 본다.
> 학습이 없어 가볍다. 단순평균 / 순위평균 / 가중평균을 비교한다.
> 예상: 강한 트리(XGB+LGBM)끼리는 소폭 오를 수 있으나, 약한 DL(MLP 0.198/TabNet 0.10)을 섞으면 떨어질 가능성.

## 0. 데이터 + 확률 로드
processed_v2의 y_val과, 각 모델이 저장한 val 확률 파일을 불러온다.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from utils import compute_metrics, log_result, load_processed

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR    = PROJECT_ROOT / "processed_v2"
NB_DIR     = PROJECT_ROOT / "notebooks_v2"
RESULTS_V2 = NB_DIR / "results_v2.csv"

_, val_df, _ = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
y_val = val_df[TARGET].values.astype(int)

xgb = np.load(NB_DIR / "xgb_v2_val_prob.npy")
lgb = np.load(NB_DIR / "lgb_v2_val_prob.npy")
mlp = np.load(NB_DIR / "mlp_v2_best_val_prob.npy")
tab = np.load(NB_DIR / "tabnet_v2_val_prob.npy")
print("로드 완료. y_val 길이:", len(y_val))

로드 완료. y_val 길이: 337572


## 1. 단일 모델 PR_AUC (기준)

In [2]:
def ap(p):
    return round(average_precision_score(y_val, p), 4)

print("xgb_v2_tuned :", ap(xgb))
print("lgb_v2_tuned :", ap(lgb))
print("mlp_v2_best  :", ap(mlp))
print("tabnet_v2    :", ap(tab))

xgb_v2_tuned : 0.2959


lgb_v2_tuned : 0.2636


mlp_v2_best  : 0.1982


tabnet_v2    : 0.1016


## 2. 블렌딩 비교
- 순위평균: 각 확률을 0~1 순위로 바꿔 평균(스케일 다른 모델 합칠 때 안전).
- 단순평균/가중평균도 비교. 기준은 단일 최고 xgb(0.296).

In [3]:
def rank(p):
    return np.argsort(np.argsort(p)) / len(p)

blends = {
    "XGB+LGB (avg)":          (xgb + lgb) / 2,
    "XGB+LGB (rank)":         (rank(xgb) + rank(lgb)) / 2,
    "XGB+LGB+MLP (rank)":     (rank(xgb) + rank(lgb) + rank(mlp)) / 3,
    "XGB+LGB+MLP+Tab (rank)": (rank(xgb) + rank(lgb) + rank(mlp) + rank(tab)) / 4,
    "XGB0.7+LGB0.3 (rank)":   0.7 * rank(xgb) + 0.3 * rank(lgb),
    "XGB0.8+LGB0.2 (rank)":   0.8 * rank(xgb) + 0.2 * rank(lgb),
}
print("단일 최고 xgb_v2_tuned:", ap(xgb), "\n")
best_name, best_p, best_ap = None, None, -1
for name, p in blends.items():
    a = ap(p)
    mark = "  <- 최고 단일 넘음" if a > ap(xgb) else ""
    print(f"  {name:26s} {a}{mark}")
    if a > best_ap:
        best_ap, best_name, best_p = a, name, p

단일 최고 xgb_v2_tuned: 0.2959 



  XGB+LGB (avg)              0.2869


  XGB+LGB (rank)             0.2857


  XGB+LGB+MLP (rank)         0.2578


  XGB+LGB+MLP+Tab (rank)     0.22


  XGB0.7+LGB0.3 (rank)       0.2905


  XGB0.8+LGB0.2 (rank)       0.2928


## 3. 최고 블렌딩 기록
블렌딩 최고가 단일 최고(xgb)를 넘으면 results_v2.csv에 기록.

In [4]:
print("블렌딩 최고:", best_name, best_ap)
if best_ap > ap(xgb):
    log_result("Ensemble_v2_best", compute_metrics(y_val, best_p), path=str(RESULTS_V2))
    np.save(NB_DIR / "ensemble_v2_val_prob.npy", best_p)
    print("-> 단일 모델을 넘어 results_v2.csv에 기록 + 확률 저장")
else:
    print("-> 단일 xgb를 못 넘음. 앙상블은 개선 없음(정직하게 기록 안 함, 또는 참고용).")
    log_result("Ensemble_v2_best", compute_metrics(y_val, best_p), path=str(RESULTS_V2))
    np.save(NB_DIR / "ensemble_v2_val_prob.npy", best_p)

블렌딩 최고: XGB0.8+LGB0.2 (rank) 0.2928


-> 단일 xgb를 못 넘음. 앙상블은 개선 없음(정직하게 기록 안 함, 또는 참고용).


## 4. 결과 비교

In [5]:
res = pd.read_csv(RESULTS_V2).drop_duplicates(subset="model", keep="last").sort_values("PR_AUC", ascending=False)
res[["model", "PR_AUC", "ROC_AUC", "Recall", "Precision", "F1"]]

,model,PR_AUC,ROC_AUC,Recall,Precision,F1
1,XGB_v2_tuned,0.2959,0.9687,0.8699,0.0861,0.1566
11,Ensemble_v2_best,0.2928,0.9688,0.9978,0.0134,0.0264
3,LGBM_v2_tuned,0.2636,0.9666,0.8907,0.0734,0.1356
0,XGB_v2_base,0.2418,0.9616,0.8911,0.0600,0.1124
2,LGBM_v2_base,0.2246,0.9568,0.8831,0.0551,0.1037
4,MLP_v2_plain_BCE,0.1982,0.9409,0.0000,0.0000,0.0000
8,MLP_v2_focal_batchnorm,0.1953,0.9476,0.1633,0.3306,0.2187
6,MLP_v2_focal,0.1952,0.9443,0.1651,0.3391,0.2221
9,MLP_v2_SMOTE,0.1924,0.9469,0.5440,0.1531,0.2389
7,MLP_v2_focal_dropout,0.1850,0.9427,0.1067,0.3691,0.1655


---
### 결론 (실행 후 해석)
- XGB+LGB 블렌딩이 단일 xgb(0.296)를 넘나? 넘으면 그게 새 최고.
- MLP/TabNet을 더하면 떨어지나? (약한 모델은 보통 앙상블을 끌어내림 - 1차 실험에서도 확인됨)
- 앙상블이 단일을 못 넘으면, "이 데이터는 단일 강한 트리가 이미 충분"이라는 정직한 결론.